# Đồ án DS300: Hệ thống gợi ý nhạc Việt

Notebook này dùng danh mục nhạc đã làm sạch gồm dữ liệu VMusic và `dataViet`. Bạn sẽ lần lượt huấn luyện và so sánh 4 phương pháp: Content-Based, Funk SVD, Item-Based CF và Popularity-Based; cuối cùng là Hybrid kết hợp cả 4. Lịch sử nghe hiện có gồm 500 tài khoản và 120.000 lượt nghe mô phỏng. Vì vậy, các chỉ số chỉ giúp kiểm tra cách chạy mô hình, chưa chứng minh chất lượng với người nghe thật.

In [2]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
search_roots = [cwd, *cwd.parents]
project_dir = next(
    (
        candidate
        for root in search_roots
        for candidate in (root, root / "ials_music_recommendation")
        if (candidate / "data_generator.py").is_file()
        and (candidate / "natural_language_search.py").is_file()
    ),
    None,
)
if project_dir is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục dự án. Hãy mở notebook trong thư mục "
        "ials_music_recommendation hoặc thư mục cha của nó."
    )
project_parent = str(project_dir.parent)
if project_parent not in sys.path:
    sys.path.insert(0, project_parent)

from ials_music_recommendation.data_generator import (
    DEFAULT_CATALOG, DEFAULT_USER_HISTORY, load_song_catalog, load_user_history,
)
from ials_music_recommendation.recommendation_service import MusicRecommendationService

songs = load_song_catalog(DEFAULT_CATALOG)
print(f"Số bài hát hợp lệ: {len(songs):,}")
print(f"Số bài có lời đã duyệt: {int(songs.has_lyrics.sum()):,}")
print("Số bài theo nguồn:")
print(songs["dataset_source"].value_counts(dropna=False))
print("\n10 thể loại phổ biến nhất:")
print(songs["genre"].value_counts(dropna=False).head(10))

Số bài hát hợp lệ: 5,459
Số bài có lời đã duyệt: 2,600
Số bài theo nguồn:
dataset_source
VMusic      4820
dataViet     639
Name: count, dtype: int64

10 thể loại phổ biến nhất:
genre
Pop                     3381
Hip hop                  333
Electronic               324
rap_viet                 157
nhac_trinh               138
rock_viet                129
cach_mang                108
tru_tinh                 107
R&B/Soul                  42
Classical/Soundtrack      30
Name: count, dtype: int64


C:\Users\Hung\AppData\Local\Temp\ipykernel_25924\2058079074.py:31: UserWarning: Skipping 1 catalog row(s) without a valid track_id or title.
  songs = load_song_catalog(DEFAULT_CATALOG)


## Gợi ý bài tương tự từ bài bạn thích

Có thể thay `SEED_TRACK_IDS` bằng mã `track_id` của một hoặc nhiều bài bạn thích. Ô bên dưới chọn sẵn một bài Việt từ `dataViet` để minh họa. Kết quả này dùng Content-Based; phần huấn luyện và đánh giá đầy đủ các mô hình nằm ở các mục phía dưới.

In [3]:
service = MusicRecommendationService(songs)

# Thay bằng track_id của những bài bạn thích.
seed_candidates = songs.loc[
    songs["dataset_source"].eq("dataViet") & songs["has_lyrics"]
]
if seed_candidates.empty:
    raise ValueError("Không có bài dataViet nào có lời đã duyệt để làm ví dụ.")
seed = seed_candidates.iloc[0]
SEED_TRACK_IDS = [str(seed["track_id"])]
print(f"Bài làm mẫu: {seed['title']} - {seed['artist']} ({SEED_TRACK_IDS[0]})")
recommendations = service.recommend_for_tracks(SEED_TRACK_IDS, top_k=10)
recommendations

Bài làm mẫu: Anh Ba Hung - Top Ca (cach_mang:anhbahung-topca_xcnk)


[{'track_id': 'cach_mang:baicanamtan-topca_ubra',
  'title': 'Bai Ca Nam Tan',
  'artist': 'Top Ca',
  'score': 0.2879443722806142,
  'genre': 'cach_mang',
  'genre_confidence': 'medium',
  'dataset_source': 'dataViet'},
 {'track_id': 'cach_mang:baonoilenroi-topcanam_czdw',
  'title': 'Bao Noi Len Roi',
  'artist': 'Top Ca Nam',
  'score': 0.2794804574202024,
  'genre': 'cach_mang',
  'genre_confidence': 'medium',
  'dataset_source': 'dataViet'},
 {'track_id': 'cach_mang:bacdangcungchungchauhanhquan-topc_3jm6j_hq',
  'title': 'Bac Dang Cung Chung Chau Hanh Quan',
  'artist': 'Top C',
  'score': 0.2790060469654883,
  'genre': 'cach_mang',
  'genre_confidence': 'medium',
  'dataset_source': 'dataViet'},
 {'track_id': 'cach_mang:baicakhongquen-quochungnsut-5129042.',
  'title': 'Bai Ca Khong Quen',
  'artist': 'Quoc Hung NSUT',
  'score': 0.261689801298211,
  'genre': 'cach_mang',
  'genre_confidence': 'medium',
  'dataset_source': 'dataViet'},
 {'track_id': 'cach_mang:baicanguoigiaovienn

## Xem thử gói ứng dụng (Content-Based và Popularity)

Mục này tải gói mô hình nhẹ dùng cho ứng dụng dòng lệnh. Gói chỉ có Content-Based, Popularity và Hybrid của hai phương pháp đó. Phần so sánh đủ bốn mô hình phía dưới được huấn luyện riêng từ tập train.

In [4]:
from ials_music_recommendation.model_bundle import TrainedModelBundle

bundle = TrainedModelBundle.from_directory(project_dir / "trained_artifacts")
hybrid_recommendations = bundle.recommend_new_user(
    liked_track_ids=SEED_TRACK_IDS,
    top_k=10,
    content_weight=0.7,
)
popular_recommendations = bundle.recommend_popular(top_k=10)
print("Gợi ý kết hợp Content-Based và Popularity:")
display(pd.DataFrame(hybrid_recommendations))
print("Bài phổ biến chung (không cá nhân hóa):")
display(pd.DataFrame(popular_recommendations))

Gợi ý kết hợp Content-Based và Popularity:


,track_id,title,artist,score,genre,genre_confidence,dataset_source
0,cach_mang:baicanamtan-topca_ubra,Bai Ca Nam Tan,Top Ca,0.011475,cach_mang,medium,dataViet
1,cach_mang:baonoilenroi-topcanam_czdw,Bao Noi Len Roi,Top Ca Nam,0.011290,cach_mang,medium,dataViet
2,cach_mang:bacdangcungchungchauhanhquan-topc_3j...,Bac Dang Cung Chung Chau Hanh Quan,Top C,0.011111,cach_mang,medium,dataViet
3,cach_mang:baicakhongquen-quochungnsut-5129042.,Bai Ca Khong Quen,Quoc Hung NSUT,0.010937,cach_mang,medium,dataViet
4,cach_mang:baicanguoigiaoviennhandan-dangduo_3c...,Bai Ca Nguoi Giao Vien Nhan Dan,Dang Duo,0.010769,cach_mang,medium,dataViet
5,cach_mang:09-rungxanhvangtiengta-lu.,Rung Xanh Vang Tieng Ta,Lu,0.010606,cach_mang,medium,dataViet
6,cach_mang:baicabencanhvong-nguyenkhanhly-24152...,Bai Ca Ben Canh Vong,Nguyen Khanh Ly,0.010448,cach_mang,medium,dataViet
7,cach_mang:11.nhanhlanrung.,Nhanh Lan Rung,nan,0.010294,cach_mang,medium,dataViet
8,cach_mang:bacdangcungchungchauhanhquan-va-2277...,Bac Dang Cung Chung Chau Hanh Quan,VA,0.010145,cach_mang,medium,dataViet
9,cach_mang:baicahanoi-trongtan_43cd.,Bai Ca Ha Noi,Trong Tan,0.010000,cach_mang,medium,dataViet


Bài phổ biến chung (không cá nhân hóa):


,track_id,title,artist,score,genre,genre_confidence,dataset_source
0,RgKAFK5djSk,Wiz Khalifa - See You Again ft. Charlie Puth F...,Wiz Khalifa,1.000000,hip hop; pittsburgh rap; pop rap; rap; souther...,medium,VMusic
1,YQHsXMglC9A,Adele - Hello,Adele,0.999375,british soul; pop; pop soul; uk pop,medium,VMusic
2,lp-EO5I60KA,Ed Sheeran - Thinking Out Loud,Ed Sheeran,0.999061,pop; uk pop,medium,VMusic
3,ApXoWvfEYVU,youtube video #ApXoWvfEYVU,Post Malone,0.998960,dfw rap; melodic rap; rap,medium,VMusic
4,3AtDnEC4zak,Charlie Puth - We Don't Talk Anymore (feat. Se...,Charlie Puth,0.998748,dance pop; pop; post-teen pop; viral pop,medium,VMusic
5,rtOvBOTyX00,Christina Perri - A Thousand Years,Christina Perri,0.998647,dance pop; neo mellow; pop; pop rock; post-tee...,medium,VMusic
6,uelHwf8o7_U,youtube video #uelHwf8o7_U,EminemMusic,0.998123,Pop,very_low,VMusic
7,DiItGE3eAyQ,youtube video #DiItGE3eAyQ,Daddy Yankee,0.998123,latin; latin hip hop; reggaeton; trap latino; ...,medium,VMusic
8,YVkUvmDQ3HY,youtube video #YVkUvmDQ3HY,EminemMusic,0.997919,Pop,very_low,VMusic
9,kXYiU_JCYtU,Numb [ UPGRADE] – Linkin Park,Linkin Park,0.997917,alternative metal; nu metal; post-grunge; rap ...,medium,VMusic


## Kiểm tra dữ liệu tài khoản mô phỏng

Tập lịch sử dưới đây là dữ liệu giả lập, không phải hành vi người nghe thật. Notebook dùng dữ liệu này để minh họa quy trình train/test và so sánh thuật toán; không dùng kết quả để khẳng định độ chính xác ngoài thực tế.

In [5]:
history = load_user_history(DEFAULT_USER_HISTORY, songs)
print(f"Số lượt nghe mô phỏng: {len(history):,}")
print(f"Số tài khoản mô phỏng: {history.user_id.nunique():,}")
print(f"Số bài có trong lịch sử: {history.song_id.nunique():,}")
history.head()

Số lượt nghe mô phỏng: 120,000
Số tài khoản mô phỏng: 500
Số bài có trong lịch sử: 4,674


,user_id,song_id,play_count,rating
0,U0001,3,8,4.0
1,U0001,6,33,4.0
2,U0001,7,19,3.0
3,U0001,11,3,2.0
4,U0001,32,1,3.0


## Tùy chọn: tạo gói Content-Based cho ứng dụng

Ô tiếp theo chỉ cần chạy khi muốn tạo lại artifacts cho ứng dụng dòng lệnh. Gói riêng này không chứa Funk SVD hoặc Item-Based CF; bốn mô hình dùng cho đồ án được huấn luyện ở phần dưới.

In [6]:
from ials_music_recommendation.train_model_artifacts import main as train_content_model

# Bỏ dấu # và chạy nếu cần tạo lại gói Content-Based cho ứng dụng.
# train_content_model()

## Bước 1: chia dữ liệu train và test

Mỗi tài khoản được chia riêng: 80% lịch sử làm tập train, 20% làm tập test (seed 42). Từ tập train, một tập validation riêng được dùng để chọn tham số (seed 43); test không được dùng để chọn mô hình hay tham số. Funk SVD học rating 1-5. Item-Based CF học các lượt nghe có rating từ 4 trở lên, với trọng số cao hơn cho bài nghe nhiều và được rating cao. Khi đánh giá, rating từ 4 trở lên được xem là phù hợp.

In [7]:
import numpy as np
from ials_music_recommendation.evaluator import df_to_matrix, evaluate, train_test_split_interactions
from ials_music_recommendation.models import (
    ContentBasedRecommender,
    FunkSVD,
    ItemBasedCF,
    PopularityBasedRecommendation,
)

RANDOM_STATE = 42
TOP_K = 10
RELEVANCE_THRESHOLD = 4

# Đổi mã tài khoản thành số nguyên để làm chỉ số hàng trong ma trận.
interactions_df = history.copy()
user_codes, user_labels = pd.factorize(interactions_df["user_id"], sort=True)
interactions_df["user_id"] = user_codes.astype(np.int64)
n_users = len(user_labels)
n_songs = len(songs)

train_df, test_df = train_test_split_interactions(
    interactions_df,
    test_ratio=0.2,
    min_interactions=5,
    random_state=RANDOM_STATE,
)
train_R = df_to_matrix(
    train_df, n_users=n_users, n_items=n_songs, value_col="rating"
)
positive_train = train_df.loc[train_df["rating"] >= RELEVANCE_THRESHOLD].copy()
if positive_train.empty:
    raise ValueError("Tập train không có tương tác nào được rating 4 hoặc 5.")

# Bài nghe nhiều và rating cao sẽ có trọng số CF lớn hơn.
positive_train["cf_strength"] = (
    positive_train["play_count"] * positive_train["rating"] / 5.0
)
cf_R = df_to_matrix(
    positive_train,
    n_users=n_users,
    n_items=n_songs,
    value_col="cf_strength",
)
seen_R = train_R.copy()

evaluated_test = test_df.loc[test_df["rating"] >= RELEVANCE_THRESHOLD]
print(f"Số tài khoản: {n_users:,} | Số bài trong danh mục: {n_songs:,}")
print(f"Số dòng train: {len(train_df):,} | Số dòng test: {len(test_df):,}")
print(f"Tương tác train rating từ 4 trở lên cho CF: {len(positive_train):,}")
print(f"Tương tác test phù hợp (rating >= {RELEVANCE_THRESHOLD}): {len(evaluated_test):,}")

[Split] Train: 96205 | Test: 23795
[Split] Users in test: 500
Số tài khoản: 500 | Số bài trong danh mục: 5,459
Số dòng train: 96,205 | Số dòng test: 23,795
Tương tác train rating từ 4 trở lên cho CF: 34,481
Tương tác test phù hợp (rating >= 4): 8,498


## Bước 2: huấn luyện từng mô hình

### Tham số đã chọn bằng validation

Các tham số dưới đây được chọn trên 500 tài khoản validation; tập test được giữ riêng. Với Item-Based CF, NDCG@10 tăng từ 0.0545 ở `k=40` lên 0.0552 ở `k=60` và không tăng thêm khi thử `k` lớn hơn, nên chọn `k=60`. Trọng số Hybrid được tìm trên 81 tổ hợp: Content-Based `{0.5, 1, 2}`, Funk SVD `{0.1, 0.25, 0.5}`, Item-Based CF `{1, 2, 3}` và Popularity-Based `{0.1, 0.25, 0.5}`.

| Mô hình | Cấu hình đã chọn |
|---|---|
| Content-Based | genre=1, text=1, lyrics=2 |
| Funk SVD | factors=16, reg=0.01, learning_rate=0.01, 4 epochs |
| Item-Based CF | k=60 |
| Popularity-Based | prior_strength=50 |
| Hybrid (RRF) | Content=2, Funk=0.1, Item=3, Popularity=0.25 |

Validation cho Hybrid: NDCG@10=0.0572, Hit Rate@10=0.434; trọng số bằng nhau đạt NDCG@10=0.0526, Hit Rate@10=0.418. Đây là kết quả chọn tham số, không phải kết quả cuối; kết quả cuối được tính ở tập test bên dưới.

### 1. Huấn luyện Content-Based

Mô hình chuyển tiêu đề, nghệ sĩ, thể loại và lời bài đã duyệt thành vector. Khi gợi ý cho tài khoản, mô hình dựa vào các bài trong tập train mà người đó rating cao.

In [8]:
print("Đang huấn luyện Content-Based từ thông tin bài hát...")
content_model = ContentBasedRecommender(
    genre_weight=1.0,
    text_weight=1.0,
    lyrics_weight=2.0,
).fit(songs)
print(f"Đã huấn luyện Content-Based cho {len(songs):,} bài.")

Đang huấn luyện Content-Based từ thông tin bài hát...
Đã huấn luyện Content-Based cho 5,459 bài.


### 2. Huấn luyện Funk SVD

Funk SVD học các vector ẩn từ rating 1-5 trong ma trận train. Bảng bên dưới cho thấy RMSE qua từng vòng huấn luyện; đây là sai số trên train, không phải kết quả cuối trên test.

In [9]:
print("Đang huấn luyện Funk SVD trên rating của tập train...")
funk_model = FunkSVD(
    factors=16,
    learning_rate=0.01,
    reg=0.01,
    n_epochs=4,
    random_state=RANDOM_STATE,
    verbose=False,
).fit(train_R)

funk_training_history = pd.DataFrame(funk_model.history).rename(columns={
    "epoch": "Vòng huấn luyện",
    "rmse": "RMSE trên train",
    "mse": "MSE trên train",
    "reg_loss": "Điều chuẩn",
    "loss": "Tổng hàm mất mát",
    "time": "Thời gian (giây)",
})
print("Đã huấn luyện Funk SVD. Lịch sử huấn luyện:")
display(funk_training_history)

Đang huấn luyện Funk SVD trên rating của tập train...
Đã huấn luyện Funk SVD. Lịch sử huấn luyện:


,Vòng huấn luyện,MSE trên train,RMSE trên train,Điều chuẩn,Tổng hàm mất mát,Thời gian (giây)
0,1,0.897906,0.947579,0.000862,0.898768,2.247642
1,2,0.890059,0.943429,0.000926,0.890985,2.400564
2,3,0.884586,0.940524,0.001011,0.885598,2.314551
3,4,0.880017,0.938092,0.001130,0.881147,2.289064


### 3. Huấn luyện Item-Based CF

Mỗi bài hát được biểu diễn bằng những tài khoản đã nghe bài đó. Mô hình tính độ tương tự cosine giữa các bài. `k=60` là số bài láng giềng tối đa dùng để tính điểm gợi ý; trên validation, đây là giá trị nhỏ nhất đạt mức NDCG@10 cao nhất trong các giá trị đã thử.

In [10]:
print("Đang huấn luyện Item-Based CF trên các lượt nghe rating cao...")
item_model = ItemBasedCF(k=60).fit(cf_R)
print(f"Đã huấn luyện Item-Based CF cho {n_songs:,} bài, k={item_model.k}.")

Đang huấn luyện Item-Based CF trên các lượt nghe rating cao...
Đã huấn luyện Item-Based CF cho 5,459 bài, k=60.


### 4. Huấn luyện Popularity-Based

Mô hình xếp hạng theo tổng lượt nghe, đồng thời làm trơn rating trung bình. Việc làm trơn giúp bài có rất ít rating không được xếp quá cao chỉ vì vài đánh giá may mắn. Mô hình này đưa ra cùng một danh sách phổ biến cho mọi tài khoản.

In [11]:
print("Đang tính điểm phổ biến từ lượt nghe và rating của tập train...")
popularity_model = PopularityBasedRecommendation(
    prior_strength=50.0,
).fit(train_df, n_items=n_songs)
print(f"Đã tính điểm cho {int(np.count_nonzero(popularity_model.scores_)):,} bài có tương tác trong train.")

Đang tính điểm phổ biến từ lượt nghe và rating của tập train...
Đã tính điểm cho 4,618 bài có tương tác trong train.


## Bước 3: tạo gợi ý và mô hình Hybrid

Hybrid kết hợp thứ hạng của bốn mô hình bằng Reciprocal Rank Fusion (RRF), nên không cộng trực tiếp các score vốn có thang đo khác nhau. Trọng số bên dưới được chọn trên validation; mỗi mô hình đóng góp 30 bài đầu vào danh sách ứng viên.

In [12]:
def recommend_funk(user_id, top_k, seen_items):
    return funk_model.recommend(
        int(user_id), top_k=top_k, seen_items=np.asarray(seen_items, dtype=np.int64)
    )


def recommend_content(user_id, top_k, seen_items):
    ranked = content_model.recommend(
        train_df,
        user_id=int(user_id),
        top_k=top_k,
        exclude_song_ids=np.asarray(seen_items, dtype=np.int64).tolist(),
    )
    return ranked["song_id"].to_numpy(dtype=np.int64)


def recommend_item(user_id, top_k, seen_items):
    return item_model.recommend(
        int(user_id), top_k=top_k, seen_items=seen_items
    )


def recommend_popularity(user_id, top_k, seen_items):
    return popularity_model.recommend(
        int(user_id), top_k=top_k, seen_items=seen_items
    )


base_recommenders = {
    "Content-Based": recommend_content,
    "Funk SVD": recommend_funk,
    "Item-Based CF": recommend_item,
    "Popularity-Based": recommend_popularity,
}


def recommend_hybrid(user_id, top_k, seen_items):
    candidate_k = min(top_k * 3, n_songs)
    weights = {
        "Content-Based": 2.0,
        "Funk SVD": 0.1,
        "Item-Based CF": 3.0,
        "Popularity-Based": 0.25,
    }
    fused_scores = {}
    for name, recommend in base_recommenders.items():
        ranked_items = recommend(user_id, candidate_k, seen_items)
        for rank, song_id in enumerate(ranked_items, start=1):
            fused_scores[int(song_id)] = fused_scores.get(int(song_id), 0.0) + (
                weights[name] / (60.0 + rank)
            )
    ranked_ids = sorted(
        fused_scores,
        key=lambda song_id: (-fused_scores[song_id], song_id),
    )
    return np.asarray(ranked_ids[:top_k], dtype=np.int64)


recommenders = {**base_recommenders, "Hybrid (RRF)": recommend_hybrid}
print("Đã sẵn sàng các mô hình:", ", ".join(recommenders))

Đã sẵn sàng các mô hình: Content-Based, Funk SVD, Item-Based CF, Popularity-Based, Hybrid (RRF)


## Bước 4: so sánh kết quả gợi ý trên tập test

Mọi mô hình được đánh giá trên cùng các tài khoản test chưa dùng khi chọn tham số, cùng ngưỡng rating từ 4 trở lên và cùng danh sách bài đã nghe trong train. NDCG@10 là độ đo chính cho thứ hạng; Precision, Recall, F1, MRR và Hit Rate là các độ đo bổ sung. Đây không phải xác suất.

In [13]:
comparison = pd.DataFrame({
    name: evaluate(
        recommend,
        seen_R,
        test_df,
        top_k=TOP_K,
        relevance_column="rating",
        relevance_threshold=RELEVANCE_THRESHOLD,
    )
    for name, recommend in recommenders.items()
}).T
comparison.index.name = "Mô hình"
comparison = comparison.rename(columns={
    "precision": f"Precision@{TOP_K} (Tỷ lệ gợi ý đúng)",
    "recall": f"Recall@{TOP_K} (Tỷ lệ tìm được)",
    "f1": f"F1@{TOP_K}",
    "ndcg": f"NDCG@{TOP_K} (Chất lượng thứ hạng)",
    "mrr": "MRR (Vị trí gợi ý đúng đầu tiên)",
    "hit_rate": f"Hit Rate@{TOP_K} (Có ít nhất một bài đúng)",
})
display(comparison.round(4))

,Precision@10 (Tỷ lệ gợi ý đúng),Recall@10 (Tỷ lệ tìm được),F1@10,NDCG@10 (Chất lượng thứ hạng),MRR (Vị trí gợi ý đúng đầu tiên),Hit Rate@10 (Có ít nhất một bài đúng)
Mô hình,,,,,,
Content-Based,0.0034,0.0021,0.0026,0.0036,0.0112,0.034
Funk SVD,0.0062,0.0037,0.0046,0.0051,0.0112,0.056
Item-Based CF,0.0888,0.0522,0.0652,0.0988,0.2580,0.598
Popularity-Based,0.0594,0.0346,0.0434,0.0681,0.2009,0.448
Hybrid (RRF),0.0906,0.0531,0.0664,0.0999,0.2598,0.624


### Đánh giá thêm độ đa dạng và nhạc ngách

`ILD@10` là độ khác nhau trung bình giữa từng cặp bài trong danh sách, tính bằng cosine distance trên vector nội dung đã chuẩn hóa; càng cao thì danh sách càng đa dạng về nội dung. `LongTailShare@10` là tỷ lệ bài nằm ngoài 20% bài phổ biến nhất, với độ phổ biến tính từ tập train. Bài không xuất hiện trong train được xếp vào long tail. Hai số đo này bổ sung cho độ liên quan, không thay thế NDCG/Precision; dữ liệu tương tác vẫn là mô phỏng.

In [14]:
from ials_music_recommendation.evaluator import evaluate_beyond_accuracy

train_item_popularity = np.bincount(
    train_df["song_id"].to_numpy(dtype=np.int64), minlength=n_songs
)
beyond_accuracy = pd.DataFrame({
    name: evaluate_beyond_accuracy(
        recommend,
        seen_R,
        evaluated_test["user_id"].unique(),
        content_model.item_features,
        train_item_popularity,
        top_k=TOP_K,
        head_fraction=0.2,
    )
    for name, recommend in recommenders.items()
}).T.rename(columns={
    "ild": f"ILD@{TOP_K} (đa dạng nội dung)",
    "long_tail_share": f"LongTailShare@{TOP_K} (tỷ lệ nhạc ngách)",
})
beyond_accuracy.index.name = "Mô hình"
display(beyond_accuracy.round(4))

,ILD@10 (đa dạng nội dung),LongTailShare@10 (tỷ lệ nhạc ngách)
Mô hình,,
Content-Based,0.6397,0.8870
Funk SVD,0.9600,0.5562
Item-Based CF,0.9516,0.0000
Popularity-Based,0.9476,0.0268
Hybrid (RRF),0.9495,0.0000


### Sai số dự đoán rating của Funk SVD

MAE và RMSE đo khoảng cách giữa rating thật và rating dự đoán trên tập test. Hai chỉ số này chỉ áp dụng cho Funk SVD.

In [15]:
test_users = test_df["user_id"].to_numpy(dtype=np.int64)
test_items = test_df["song_id"].to_numpy(dtype=np.int64)
actual_ratings = test_df["rating"].to_numpy(dtype=np.float64)
predicted_ratings = np.asarray([
    funk_model.predict(int(user_id), int(song_id))
    for user_id, song_id in zip(test_users, test_items)
])
# Giới hạn rating dự đoán trong khoảng 1-5 khi tính sai số.
predicted_ratings = np.clip(predicted_ratings, 1.0, 5.0)
rating_errors = actual_ratings - predicted_ratings
rating_error_metrics = pd.DataFrame(
    {"Funk SVD": {
        "MAE": float(np.mean(np.abs(rating_errors))),
        "RMSE": float(np.sqrt(np.mean(rating_errors ** 2))),
    }}
)
print("Sai số rating trên tập test (càng thấp càng tốt; dữ liệu mô phỏng):")
display(rating_error_metrics.round(4))

Sai số rating trên tập test (càng thấp càng tốt; dữ liệu mô phỏng):


,Funk SVD
MAE,0.7665
RMSE,0.9701


## Bước 5: xem gợi ý cho một tài khoản mô phỏng

Mỗi lần chạy ô dưới, notebook chọn ngẫu nhiên một tài khoản trong dữ liệu mô phỏng và hiển thị kết quả của năm phương pháp. Muốn xem một tài khoản cụ thể, thay `USER_TO_DEMO` bằng số từ `0` đến `n_users - 1`.

In [16]:
rng = np.random.default_rng()
USER_TO_DEMO = int(rng.choice(n_users))
original_user_id = str(user_labels[USER_TO_DEMO])
seen_items = seen_R.getrow(USER_TO_DEMO).indices
print(f"Tài khoản mô phỏng: {original_user_id} (hàng ma trận {USER_TO_DEMO})")


def recommendation_table(song_ids):
    result = songs.set_index("song_id").loc[list(song_ids)].reset_index()
    columns = [
        column for column in
        ("song_id", "track_id", "title", "artist", "genre", "dataset_source")
        if column in result.columns
    ]
    return result[columns]


for model_name, recommend in recommenders.items():
    song_ids = recommend(USER_TO_DEMO, TOP_K, seen_items)
    print(f"\nGợi ý từ mô hình: {model_name}")
    display(recommendation_table(song_ids))
print("\nCác bài tài khoản này đã nghe:")
account_history = interactions_df.loc[
    interactions_df["user_id"].eq(USER_TO_DEMO),
    ["song_id", "rating", "play_count"],
].copy()
test_song_ids = set(
    test_df.loc[test_df["user_id"].eq(USER_TO_DEMO), "song_id"].tolist()
)
account_history["Tập dữ liệu"] = np.where(
    account_history["song_id"].isin(test_song_ids),
    "Test (chỉ đánh giá)",
    "Train",
)
song_details = songs[[
    column for column in
    ("song_id", "track_id", "title", "artist", "genre", "dataset_source")
    if column in songs.columns
]]
account_history = account_history.merge(
    song_details,
    on="song_id",
    how="left",
    validate="many_to_one",
).sort_values(
    ["Tập dữ liệu", "rating", "play_count"],
    ascending=[True, False, False],
).reset_index(drop=True)
history_columns = [
    column for column in
    ("song_id", "track_id", "title", "artist", "genre", "dataset_source",
     "rating", "play_count", "Tập dữ liệu")
    if column in account_history.columns
]
display(account_history[history_columns])
print(f"Tổng số bài đã nghe: {len(account_history)}")

Tài khoản mô phỏng: U0352 (hàng ma trận 351)

Gợi ý từ mô hình: Content-Based


,song_id,track_id,title,artist,genre,dataset_source
0,3019,gJYT4Uggy_U,Từng Là Của Nhau,Bảo Anh,Pop,VMusic
1,1861,8oa9i2_p7K4,Thật Tuyệt Vời Khi Ở Bên Em / OST #TTVKOBE - A...,Andiez Music,Pop,VMusic
2,1642,AiD1a2fFFLw,KHÔNG SAO MÀ EM ĐÂY RỒI | SUNI HẠ LINH ft. Lou...,SUNI,Pop,VMusic
3,4682,Oo-bkxvlAUk,Em ! Có Yêu Anh Không - Khánh Phương [MV OFFIC...,Nam Việt Music,Pop,VMusic
4,3872,0L9umD7Sf8Y,LONELINESS | Đinh Hương ft BigDaddy | Nhạc trẻ...,YEAH1 MUSIC,Pop,VMusic
5,1732,cUmpJ2zwfVU,SUÝT NỮA THÌ | OFFICIAL OST | CHUYẾN ĐI CỦA TH...,Andiez Music,Pop,VMusic
6,3817,ioHp2iGUwyg,Còn Thương Thì Không Để Em Khóc - Miu Lê x Đạt...,Miu Lê,Pop,VMusic
7,3831,J1Cs6URbH0o,Sẽ Có Người Cần Anh - Cao Thái Sơn x Hương Tràm,POPS MUSIC,Pop,VMusic
8,2955,jDC1tCdq7mQ,Yêu Em Hơn Mỗi Ngày,Andiez Music,Pop,VMusic
9,2710,l95OmLzZNGg,MÃI MÃI KHÔNG PHẢI ANH,Thanh Bình,Pop,VMusic



Gợi ý từ mô hình: Funk SVD


,song_id,track_id,title,artist,genre,dataset_source
0,3945,K8wtSelj6WQ,NẾU TA KHÔNG MAY | Anh Tú ft LyLy,Anh Tú,Pop,VMusic
1,1243,V6Iiuz_3rsE,youtube video #V6Iiuz_3rsE,Avicii,big room; dance pop; edm; pop; pop dance,VMusic
2,42,k2C5TjS2sh4,Roxette - It Must Have Been Love,Roxette,dance pop; europop; new wave pop; pop rock; so...,VMusic
3,1513,i_kF4zLNKio,youtube video #i_kF4zLNKio,Harlem Fetty,Pop,VMusic
4,2036,18aXiy8xvSw,ATBA - HIDE THIS LOVE,Big Arts Entertainment,Pop,VMusic
5,50,zRIbf6JqkNc,Guns N' Roses - Don't Cry,Guns N' Roses,glam metal; hard rock; rock,VMusic
6,205,X1IC4U9tFdc,Fella Sleep - You Got Me,We Are Diamond,Pop,VMusic
7,494,Mx_OexsUI2M,Rihanna - Lift Me Up (From Black Panther: Waka...,Rihanna,barbadian pop; dance pop; pop; pop dance; post...,VMusic
8,3999,SMyzXbdGxVY,tlinh - nữ siêu anh hùng,tlinh,Pop,VMusic
9,2498,lz6qJRlsyd0,Đại Minh Tinh,VĂN MAI HƯƠNG,Pop,VMusic



Gợi ý từ mô hình: Item-Based CF


,song_id,track_id,title,artist,genre,dataset_source
0,4520,qAHxvhGk_mQ,Bức Tranh Từ Nước Mắt - Mr. Siro [ 1080p],Mr. Siro,Pop,VMusic
1,2900,nQ1BG2fHef4,Sau Lời Từ Khước,Release - Topic,Pop,VMusic
2,3900,7BmpKNCJphE,THE MEN | PHẢI ĐÀNH QUÊN THÔI,The Men,Pop,VMusic
3,3063,pHWHVNkp4OY,EM LÀ KẺ ĐÁNG THƯƠNG,Phát Huy T4,Pop,VMusic
4,3981,p8A3ULXgV3E,Nắm Lấy Tay Anh - Tuấn Hưng [MV Official],Tuấn Hưng,Pop,VMusic
5,3503,quGW29NiblY,Chạy Theo Lý Trí - Hồ Ngọc Hà,Ho Ngoc Ha's,Pop,VMusic
6,1925,FFkSXNmuy6o,"từ thích thích , thành thương thương - amee ft...",Nhạc Này Chill Phết,Pop,VMusic
7,4377,3K4WC1vw8Is,Ngày Xưa Em Đến | Anh Khang,POPS MUSIC,Pop,VMusic
8,1375,Dst9gZkq1a8,youtube video #Dst9gZkq1a8,Travis Scott,rap,VMusic
9,605,ZEWGyyLiqY4,Blake Shelton - God's Country,Blake Shelton,contemporary country; country; country road; o...,VMusic


... (3 outputs left out)

,song_id,track_id,title,artist,genre,dataset_source
0,4520,qAHxvhGk_mQ,Bức Tranh Từ Nước Mắt - Mr. Siro [ 1080p],Mr. Siro,Pop,VMusic
1,2900,nQ1BG2fHef4,Sau Lời Từ Khước,Release - Topic,Pop,VMusic
2,3900,7BmpKNCJphE,THE MEN | PHẢI ĐÀNH QUÊN THÔI,The Men,Pop,VMusic
3,3063,pHWHVNkp4OY,EM LÀ KẺ ĐÁNG THƯƠNG,Phát Huy T4,Pop,VMusic
4,3503,quGW29NiblY,Chạy Theo Lý Trí - Hồ Ngọc Hà,Ho Ngoc Ha's,Pop,VMusic
5,3981,p8A3ULXgV3E,Nắm Lấy Tay Anh - Tuấn Hưng [MV Official],Tuấn Hưng,Pop,VMusic
6,1925,FFkSXNmuy6o,"từ thích thích , thành thương thương - amee ft...",Nhạc Này Chill Phết,Pop,VMusic
7,4377,3K4WC1vw8Is,Ngày Xưa Em Đến | Anh Khang,POPS MUSIC,Pop,VMusic
8,605,ZEWGyyLiqY4,Blake Shelton - God's Country,Blake Shelton,contemporary country; country; country road; o...,VMusic
9,2534,Q-NzYzzx65c,Chiều Hôm Ấy,JayKii,Pop,VMusic



Các bài tài khoản này đã nghe:


,song_id,track_id,title,artist,genre,dataset_source,rating,play_count,Tập dữ liệu
0,3331,zLAEYej8TfM,Vy Oanh - ĐỒNG XANH,ifocusvn,Pop,VMusic,5.0,13,Test (chỉ đánh giá)
1,4249,QFZEqDdzoPo,[2G17] Mấy con mèo - Datmaniac,G - FAMILY,Pop,VMusic,5.0,12,Test (chỉ đánh giá)
2,2459,7P9LerF-f8U,Có Đau Cũng Là Xứng Đáng - Quốc Thiên | | OST ...,Quốc Thiên,Pop,VMusic,5.0,7,Test (chỉ đánh giá)
3,2140,Rd2vmaHyjxw,HAN SARA - I’M YOUR CHRISTMAS PRESENT | Offici...,Han Sara,Electronic,VMusic,5.0,6,Test (chỉ đánh giá)
4,1245,xP-lPpRezYM,youtube video #xP-lPpRezYM,Dimitri Vegas & Like Mike,belgian dance; belgian edm; big room; edm; ele...,VMusic,5.0,3,Test (chỉ đánh giá)
...,...,...,...,...,...,...,...,...,...
236,1317,ROAtsHmvJhM,youtube video #ROAtsHmvJhM,Marshmello,brostep; edm; pop; pop dance; progressive elec...,VMusic,1.0,12,Train
237,1116,MFUzM6nG5Vk,[classical guitar]NGƯỜI ƠI NGƯỜI Ở ĐỪNG VỀ by ...,TS.Phúc,Classical/Soundtrack,VMusic,1.0,4,Train
238,3593,2GBJCmXQ0Lk,Không Thể Quên | Tronie Ngô | Yeah1 Superstar ...,YEAH1 MUSIC,Pop,VMusic,1.0,3,Train
239,4705,5PvzGDU8WjM,HIEUTHUHAI - Không Thể Say | Live Performance ...,HIEUTHUHAI,Pop,VMusic,1.0,3,Train


Tổng số bài đã nghe: 241


## Thử tìm nhạc theo tâm trạng bằng câu mô tả

Bản demo offline nhận diện các nhóm tâm trạng `buồn`, `vui`, `chill`, `lãng mạn`, `năng lượng`, `hoài niệm`, `sâu lắng` và `tập trung`, cùng các cách diễn đạt gần nghĩa, thể loại và nghệ sĩ có trong catalog. Kết quả được xếp hạng bằng TF-IDF trên metadata và lời bài đã được duyệt; nghệ sĩ được nhắc đến sẽ được ưu tiên nhẹ. Đây là tìm kiếm từ khóa có mở rộng từ, chưa phải mô hình hiểu ngữ nghĩa/LLM và không kết luận chính xác cảm xúc của bài hát.

In [17]:
from pathlib import Path
import sys

# Cho phép chạy riêng ô này, kể cả khi ô thiết lập đầu notebook chưa chạy.
cwd = Path.cwd().resolve()
search_roots = [cwd, *cwd.parents]
project_dir = next(
    (
        candidate
        for root in search_roots
        for candidate in (root, root / "ials_music_recommendation")
        if (candidate / "natural_language_search.py").is_file()
        and (candidate / "data_generator.py").is_file()
    ),
    None,
)
if project_dir is None:
    raise FileNotFoundError(
        "Không tìm thấy mã dự án. Hãy mở notebook trong thư mục "
        "ials_music_recommendation hoặc thư mục cha của nó."
    )
project_parent = str(project_dir.parent)
if project_parent not in sys.path:
    sys.path.insert(0, project_parent)

from ials_music_recommendation.natural_language_search import NaturalLanguageMusicSearch

mood_search = NaturalLanguageMusicSearch(songs)
MUSIC_QUERY = "nhạc tình yêu"
print("Câu tìm:", MUSIC_QUERY)
print("Thông tin nhận diện:", mood_search.parse_query(MUSIC_QUERY))
mood_results = mood_search.search(MUSIC_QUERY, top_k=10)
display(mood_results[[
    column for column in ("track_id", "title", "artist", "genre", "score")
    if column in mood_results.columns
]])

Câu tìm: nhạc tình yêu
Thông tin nhận diện: {'moods': ['lãng mạn'], 'genres': [], 'artists': []}


,track_id,title,artist,genre,score
0,CVx2jS_6nxo,Người Em Tìm Kiếm - MIN (OFFICIAL M/V),1989s Entertainment,Pop,0.216446
1,7ym_-52w65M,Dĩ Vãng Nhạt Nhòa (Remake),Hà Nhi,Pop,0.212947
2,ZJWye6vG3vY,ĐÔNG NHI - NGỌT NGÀO,ĐÔNG NHI,Pop,0.211908
3,tru_tinh:06.doantuyet.,Doan Tuyet,,tru_tinh,0.200894
4,h6AaGjs4VR4,"Hẹn Một Mai (From"" 4 Năm 2 Chàng 1 Tình Yêu"")",Bùi Anh Tuấn,Pop,0.195388
5,xvpverLphlo,Ngọt - Đốt,Ngọt,Pop,0.191634
6,S3YvgwfxJ68,Nguyễn Trần Trung Quân – Trong Trí Nhớ Của Anh,Nguyễn Trần Trung Quân,Electronic,0.190999
7,nhac_trinh:tinhsau-thudang-2596277.,Tinh Sau,Thu Dang,nhac_trinh,0.188810
8,69Jldtu2zWw,Năm Ấy,ĐỨC PHÚC,Pop,0.186086
9,pX6nutvtDnI,Hẹn Một Mai | Bùi Anh Tuấn | | (OST 4 Năm 2 Ch...,YEAH1 MUSIC,Pop,0.186000


### Cách hiểu kết quả và giới hạn

- Kết quả trên test: Hybrid đạt NDCG@10=0.0999, Hit Rate@10=0.624 và Precision@10=0.0906; Item-Based CF đứng kế tiếp với NDCG@10=0.0988. So với cấu hình Hybrid cũ trên cùng test (NDCG@10=0.0746, Hit Rate@10=0.496), NDCG tăng khoảng 34% và Hit Rate tăng 12.8 điểm phần trăm. Các chỉ số này đến từ lịch sử mô phỏng, không chứng minh chất lượng với người nghe thật.
- Funk SVD dự đoán rating từ 1 đến 5. Cấu hình hiện tại ưu tiên xếp hạng top-10; MAE=0.7665 và RMSE=0.9701 trên test. Nếu mục tiêu chính là dự đoán rating, cần chọn tham số riêng theo validation MAE/RMSE.
- Content-Based tìm bài giống các bài tài khoản đánh giá cao. Item-Based CF tìm bài có mẫu lịch sử nghe tương tự. Popularity xếp hạng chung theo lượt nghe và rating. Hybrid kết hợp thứ hạng của cả bốn mô hình.
- Toàn bộ lịch sử hiện có là dữ liệu mô phỏng. Không nên trình bày các chỉ số này như độ chính xác trên người nghe thật.
- Gói ứng dụng ở phần đầu notebook chỉ có Content-Based và Popularity. Các ô huấn luyện bằng `FunkSVD.fit`, `ItemBasedCF.fit` và `PopularityBasedRecommendation.fit` nằm riêng trong notebook này.

## Thử nghiệm emotion-aware trên dataViet

Phần này dùng Ollama local trên 335 bài dataViet có lời đã duyệt; đây là thử nghiệm riêng, không thay thế Content-Based chính. Tạo hoặc sửa summary bằng `python -m ials_music_recommendation.emotion_pipeline --max-items 335 --dataset-source dataViet --repair-truncated` từ thư mục cha dự án.

In [18]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_dir = next(
    (
        candidate
        for root in [cwd, *cwd.parents]
        for candidate in (root, root / "ials_music_recommendation")
        if (candidate / "emotion_artifacts").is_dir()
        and (candidate / "data_generator.py").is_file()
    ),
    None,
)
if project_dir is None:
    raise FileNotFoundError(
        "Chưa tìm thấy emotion_artifacts. Hãy chạy pipeline theo hướng dẫn "
        "trong APP_HANDOFF_README.md trước."
    )
project_parent = str(project_dir.parent)
if project_parent not in sys.path:
    sys.path.insert(0, project_parent)

from ials_music_recommendation.data_generator import load_song_catalog
from ials_music_recommendation.models.emotion_aware import EmotionAwareRecommender

songs = load_song_catalog()
emotion_model = EmotionAwareRecommender.from_artifacts(
    songs, project_dir / "emotion_artifacts", emotion_weight=0.5
)
sample_seed = emotion_model.songs.iloc[0]
sample_seed_id = sample_seed["song_id"]
baseline_results = emotion_model.baseline.recommend_from_items(
    [sample_seed_id], top_k=10
)
emotion_results = emotion_model.recommend_from_items(
    [sample_seed_id], top_k=10
)
overlap = len(
    set(baseline_results["track_id"]) & set(emotion_results["track_id"])
)
print(f"Bài làm mẫu: {sample_seed['title']} - {sample_seed['artist']}")
print(f"Số bài trùng trong top-10: {overlap}/10")
display(emotion_results[[
    "track_id", "title", "artist", "score", "content_score", "emotion_score"
]])

C:\Users\Hung\AppData\Local\Temp\ipykernel_25924\896180719.py:27: UserWarning: Skipping 1 catalog row(s) without a valid track_id or title.
  songs = load_song_catalog()


Bài làm mẫu: Anh Ba Hung - Top Ca
Số bài trùng trong top-10: 7/10


,track_id,title,artist,score,content_score,emotion_score
0,cach_mang:baicanamtan-topca_ubra,Bai Ca Nam Tan,Top Ca,0.582919,0.267337,0.898501
1,cach_mang:baonoilenroi-topcanam_czdw,Bao Noi Len Roi,Top Ca Nam,0.563964,0.264683,0.863245
2,cach_mang:bacdangcungchungchauhanhquan-topc_3j...,Bac Dang Cung Chung Chau Hanh Quan,Top C,0.558414,0.261358,0.855469
3,cach_mang:baicahyvong-lananh_9fx_hq,Bai Ca Hy Vong,Lan Anh,0.554785,0.231488,0.878081
4,cach_mang:baicakhongquen-quochungnsut-5129042.,Bai Ca Khong Quen,Quoc Hung NSUT,0.554120,0.228412,0.879829
5,cach_mang:baicathongnhat-anhtho_38fnf.,Bai Ca Thong Nhat,Anh Tho,0.551975,0.225968,0.877982
6,cach_mang:baicahyvong-tungduong-3826024_hq,Bai Ca Hy Vong,Tung Duong,0.545848,0.213358,0.878338
7,cach_mang:baicanguoigiaoviennhandan-dangduo_3c...,Bai Ca Nguoi Giao Vien Nhan Dan,Dang Duo,0.538649,0.229503,0.847795
8,cach_mang:baicakhongquen-taminhtam-2267775.,Bai Ca Khong Quen,Ta Minh Tam,0.538582,0.207777,0.869388
9,cach_mang:baicanamtan-nsndthuhien_326tk,Bai Ca Nam Tan,NSND Thu Hien,0.536924,0.197138,0.876711


### So sánh trọng số và phiếu đánh giá nhóm

So sánh weight 0, 0.25, 0.5, 0.75 và 1 trên cùng 335 seed. Các metric đo đa dạng, long-tail và mức trùng baseline, không phải độ chính xác. Chạy ô kế tiếp một lần; các phiếu đánh giá được lưu trong `emotion_artifacts/experiment/`.

In [19]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_dir = next(
    (
        candidate
        for root in [cwd, *cwd.parents]
        for candidate in (root, root / "ials_music_recommendation")
        if (candidate / "emotion_artifacts").is_dir()
        and (candidate / "data_generator.py").is_file()
    ),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Chưa tìm thấy emotion_artifacts trong dự án.")
project_parent = str(project_dir.parent)
if project_parent not in sys.path:
    sys.path.insert(0, project_parent)

import pandas as pd
from ials_music_recommendation.emotion_experiment import run_experiment

experiment_report = run_experiment(
    artifact_dir=project_dir / "emotion_artifacts",
    output_dir=project_dir / "emotion_artifacts" / "experiment",
)
weight_results = pd.read_csv(
    project_dir / "emotion_artifacts" / "experiment" / "weight_comparison.csv"
)
display(weight_results)

D:\Học tập\DS300\Đồ án\ials_music_recommendation\emotion_experiment.py:124: UserWarning: Skipping 1 catalog row(s) without a valid track_id or title.
  catalog = load_song_catalog()


[COMPARE] 50/335 seeds
[COMPARE] 100/335 seeds
[COMPARE] 150/335 seeds
[COMPARE] 200/335 seeds
[COMPARE] 250/335 seeds
[COMPARE] 300/335 seeds
Compared 5 weights across 335 dataViet seeds.
[WARN] LongTailShare is unavailable: dataViet has no varying view-count or interaction popularity signal.
 emotion_weight  n_seeds  baseline_top10_overlap_mean  ild_at_10_mean  long_tail_share_at_10_mean  mean_emotion_similarity  unique_artists_at_10_mean
           0.00      335                     1.000000        0.663806                         NaN                 0.853929                   8.023881
           0.25      335                     0.912239        0.668340                         NaN                 0.857194                   7.970149
           0.50      335                     0.776418        0.678033                         NaN                 0.862210                   7.979104
           0.75      335                     0.584478        0.698442                         NaN        

,emotion_weight,n_seeds,baseline_top10_overlap_mean,ild_at_10_mean,long_tail_share_at_10_mean,mean_emotion_similarity,unique_artists_at_10_mean
0,0.00,335,1.000000,0.663806,NaN,0.853929,8.023881
1,0.25,335,0.912239,0.668340,NaN,0.857194,7.970149
2,0.50,335,0.776418,0.678033,NaN,0.862210,7.979104
3,0.75,335,0.584478,0.698442,NaN,0.868277,7.976119
4,1.00,335,0.294030,0.770642,NaN,0.873834,8.059701


### Chấm bằng metric local (miễn phí)

Pilot Qwen-3B cho điểm gần như giống nhau nên không dùng LLM để tự chấm. Điểm dưới dùng cosine BGE-M3 giữa các summary cảm xúc, TF-IDF similarity và ILD@10. Thang 1–5 là thứ hạng tương đối giữa 5 danh sách cùng seed (3 là hòa), không phải điểm chất lượng tuyệt đối. Chấm bằng artifacts local, không cần API key hoặc bật Ollama. Summary vẫn cần người kiểm tra; cosine chỉ là phép đo ngữ nghĩa.

In [20]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_dir = next(
    (candidate for root in [cwd, *cwd.parents]
     for candidate in (root, root / "ials_music_recommendation")
     if (candidate / "emotion_artifacts").is_dir()
     and (candidate / "emotion_judge.py").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("emotion_judge.py was not found in the project.")
project_parent = str(project_dir.parent)
if project_parent not in sys.path:
    sys.path.insert(0, project_parent)

from ials_music_recommendation.emotion_judge import run_judgement

RUN_LOCAL_METRICS = False  # Set True to compute local metrics.
SCORE_ALL_SEEDS = False  # Keep False for one seed; set True for all 20 seeds.
if RUN_LOCAL_METRICS:
    judge_scores = run_judgement(
        artifact_dir=project_dir / "emotion_artifacts",
        experiment_dir=project_dir / "emotion_artifacts" / "experiment",
        max_seeds=None if SCORE_ALL_SEEDS else 1,
    )
    display(judge_scores.head(25))

### Mở khóa và tổng hợp (chỉ sau khi đã chấm xong)

Chỉ bật ô dưới sau khi đã có đủ 100 điểm cho 100 danh sách hiện tại. Hàm kiểm tra cả chữ ký dữ liệu để từ chối điểm cũ/lệch danh sách trước khi đọc bảng giải mã.

In [21]:
from ials_music_recommendation.emotion_judge import summarize_by_weight

UNBLIND_LOCAL_METRICS = False  # Set True only when all lists are scored.
if UNBLIND_LOCAL_METRICS:
    judge_summary = summarize_by_weight(
        experiment_dir=project_dir / "emotion_artifacts" / "experiment"
    )
    display(judge_summary)